# HIFA — Train trên Kaggle (GPU T4)

Notebook chạy A1 (FLAG-Trader-style, Qwen2.5-1.5B + LoRA + PPO) trên GPU T4 của Kaggle.

## ⚠ﺏ TRƯỚC KHI CHẤY — 2 cài đặt BẮT BUỘC trong panel phải (Settings):
1. **Accelerator → GPU T4 x2** (hoặc P100).
2. **Internet → ON** (cần để tải Qwen2.5-1.5B từ HuggingFace + fetch dữ liệu yfinance).

## Chống mất tiến độ
- Checkpoint lưu vào `/kaggle/working/` — **được Kaggle giữ lại qua các lần Save Version** (commit).
- Script tự resume mỗi 10 update. Phiên Kaggle tối đa ~12h — nếu chưa xong, **Save Version → mở lại → chạy tiếp** (tự resume).
- Dùng config `configs/phase_a_t4.yaml` (nhẹ hơn, tối ưu cho T4).

## 1. Kiểm tra GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'KHÔNG thấy GPU! Settings → Accelerator → GPU T4 x2 rồi chạy lại.'
print('CUDA available:', torch.cuda.is_available())
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Lấy code
**Cách A (khuyến nghị) — clone từ GitHub:** sửa `<username>` thành repo của bạn.

**Cách B — upload code lên Kaggle Dataset:** nếu repo private, nén thư mục `hifa/` thành zip, tạo Dataset, Add Data vào notebook, rồi `!cp -r /kaggle/input/<ten-dataset>/hifa /kaggle/working/` (bỏ qua cell clone bên dưới).

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/hifa'):
    !git clone https://github.com/<username>/hifa.git
os.chdir('/kaggle/working/hifa')
os.environ['PYTHONPATH'] = '/kaggle/working/hifa'
print('cwd =', os.getcwd())

## 3. Trỏ checkpoints/ + logs/ về /kaggle/working (bền vững qua các Version)
Code clone vào `/kaggle/working/hifa` — bản thân nó đã nằm trong vùng persistent của Kaggle,
nên checkpoint ghi vào `hifa/checkpoints` sẽ được giữ khi Save Version. Cell này chỉ tạo sẵn thư mục.

In [ ]:
import os
for sub in ('checkpoints/phase_a', 'logs', 'reports/phase_a'):
    os.makedirs(f'/kaggle/working/hifa/{sub}', exist_ok=True)
print('Đã tạo thư mục checkpoints/logs/reports.')

## 4. Cài dependencies (KHÔNG cài lại torch)
Kaggle cài sẵn torch+CUDA. Chỉ báo kỹ thuật tự implement bằng pandas thuần nên không cần pandas-ta.

In [ ]:
!pip install -q transformers peft accelerate \
    gymnasium yfinance quantstats \
    pyyaml scipy pyarrow
# torchao cũ có thể làm PEFT lỗi khi inject LoRA -> gỡ nếu có (ta không dùng quantization).
!pip uninstall -y -q torchao 2>/dev/null || true
import pandas, numpy
print('pandas:', pandas.__version__, '| numpy:', numpy.__version__)

## 5. Lấy dữ liệu Mỹ (fetch qua yfinance, cần Internet ON, ~2 phút)

In [ ]:
# Bỏ qua nếu đã có dữ liệu từ lần chạy trước (persistent).
import os
if not os.path.exists('/kaggle/working/hifa/data/processed/phase_0.parquet'):
    !python -m src.data.fetch_ohlcv_us --phase 0 --benchmark both
    !python -m src.data.build_dataset --phase 0
else:
    print('Đã có data/processed/phase_0.parquet — bỏ qua fetch.')

## 6. Smoke test LLM thật trên GPU (verify model lên GPU + đo tốc độ)

In [ ]:
# Phải thấy: 'policy device = cuda:0' và forward() nhanh (~0.05-0.2s).
!python scripts/_smoke_test_llm_real.py

## 7. Train A1 — 1 seed một lần, TỰ RESUME (config T4)
**Khuyến nghị:** chạy TẮNG SEED MỘT (đổi `SEED` rồi Run), vì mỗi seed mất vài giờ và phiên Kaggle giới hạn ~12h.
Nếu phiên hết giờ giữa chừng: **Save Version** rồi mở lại, chạy lại cell này — nó tự resume từ update dở.

Đầu tiên hãy chạy thử ~20 update rồi xem metrics (approx_kl < 0.1, clip_frac < 0.3) trước khi chạy đủ 200.

In [ ]:
SEED = 0  # đổi thành 1, 2, 3, 4 cho các seed sau
!python -m src.training.train_phase_a --config configs/phase_a_t4.yaml --seed {SEED}

## 8. (Tùy chọn) Theo dõi log train
Xem nhanh 20 dòng cuối của history seed hiện tại.

In [ ]:
import json, glob
for f in sorted(glob.glob('/kaggle/working/hifa/checkpoints/phase_a/seed_*/train_history.json')):
    h = json.load(open(f))
    print(f, '| updates:', len(h))
    if h:
        last = h[-1]
        print('   last:', {k: round(v, 5) for k, v in last.items() if isinstance(v, (int, float))})

## 9. Lưu kết quả
Checkpoint + logs đã nằm trong `/kaggle/working/hifa/` (persistent). Bấm **Save Version** (Quick Save)
ở góc phải để giữ lại cho lần sau. Download: tab **Output** của notebook version.

Các file quan trọng mỗi seed:
- `checkpoints/phase_a/seed_N/resume.pt` — checkpoint resume (weights + optimizer + vị trí update).
- `checkpoints/phase_a/seed_N/policy_trainable.pt` — checkpoint cuối (dùng cho evaluation).
- `checkpoints/phase_a/seed_N/train_history.json` — lịch sử metrics mọi update.

In [ ]:
!ls -R /kaggle/working/hifa/checkpoints/phase_a 2>/dev/null || echo 'Chưa có checkpoint.'